# Дообучение REVE v2 на NMT → проверка на турецком наборе: мягче обучение, честный порог, оценки по окнам

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-23 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Перезапуск дообучения REVE (`brain-bzh/reve-base`): в прошлом блокноте (`h1_2026-10-21_zs_reve-finetune`) лучшая эпоха во всех 3 запусках — первая, модель быстро переучивается. Здесь — выбор более мягкой скорости основы, остановка на плато, порог по отложенным 15 % NMT train, сохранение оценок каждого окна |
| **Данные** | как в прошлом блокноте: NMT train (`mhmdrdwn/nmt-eeg`), до 8 окон на запись, 15 % записей (тот же раздел, `random_state=0`) — выбор настройки, эпохи и порога. Проверка один раз в конце: NMT eval и турецкий набор (`buraktaci/turkish-epilepsy`) |
| **Вход** | 200 Гц, 19 каналов, окна 10 с, каждое окно — к общему СКО 1 по всем каналам; фильтр 0,3–75 Гц + режекция 50 Гц. NMT eval и Турция — до 60 окон на запись равномерно. Отложенные 15 % для порога и выбора способа сборки — тоже до 60 окон равномерно (перечитываются после раздела), для выбора эпохи — те же 8 окон, что в прошлом блокноте |
| **Аугментации** | как раньше: шум 0,1·СКО, зануление 1–2 каналов (p = 0,5), зеркальная замена полушарий (p = 0,5), сдвиг до 2 с |
| **Обучение** | голова 5·10⁻⁴; основа — по отложенным 15 %: **1·10⁻⁵** (контроль, прежнее значение), **3·10⁻⁶**, и, если хватает времени, **1·10⁻⁵ с замороженной первой половиной** (вложения + слои 1–11 из 22). До 8 эпох, остановка, если отложенный ROC AUC не растёт 2 эпохи; берётся лучшая эпоха. Затем 3 запуска выбранной настройки (запуск 0 — из выбора). Косинус с разогревом 5 % на 8 эпох, пакет 32, fp16 с пропуском шагов с NaN/inf |
| **Оценка записи** | медиана вероятностей окон (как в прошлом блокноте REVE); другие способы сборки — локально по csv окон (раздел 7) |
| **Бюджет** | ≤ 1,5 ч GPU T4; если не укладывается — сокращается перебор, а не число запусков |
| **Среда** | Kaggle, GPU T4 |

In [ ]:
import os, sys, glob, time
SMOKE = False   # True — короткая проверка на процессоре Kaggle (часть записей, ≤2 эпохи), её цифры не смотреть
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, mne, scipy.io, scipy.signal as ss
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from scipy.stats import rankdata, spearmanr
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel
mne.set_log_level("ERROR")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = "/kaggle/working"
T_NB = time.time()
END = T_NB + (45 if SMOKE else 84) * 60      # весь блокнот — не дольше 84 мин (бюджет 1,5 ч GPU с запасом на запуск сеанса)
left = lambda: (END - time.time()) / 60

SFREQ, SEG = 200, 10
L = SFREQ * SEG
CAP_TRAIN, CAP_EVAL = 8, 60

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
NEW = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True)
try:
    POS = pos_bank([NEW.get(e, e) for e in EL])
except Exception as e:
    print("новые названия не подошли:", e); POS = pos_bank(EL)
POS = POS.to(DEV)
print("позиции:", tuple(POS.shape))

MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
MIRROR = [EL.index(MIRROR_PAIRS.get(e, e)) for e in EL]

def rv_input(W):
    """Окно → общее СКО 1 по всем каналам и отсчётам (абсолютная амплитуда пропадает, соотношение каналов остаётся), обрезка ±15."""
    x = W.astype(np.float32)
    x = x - x.mean(-1, keepdims=True)
    x = x / (x.std(axis=(1, 2), keepdims=True) + 1e-6)
    return torch.from_numpy(np.clip(x, -15, 15))

class RVNet(nn.Module):
    """REVE (22 слоя) + голова: среднее по времени с сохранением каналов -> 19*512 -> два линейных слоя."""
    def __init__(self, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(19 * 512, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        o = self.backbone(x, POS.expand(x.size(0), -1, -1))
        if isinstance(o, (tuple, list)): o = o[0]
        if hasattr(o, "last_hidden_state"): o = o.last_hidden_state
        assert o.dim() == 4 and o.shape[1] == 19, tuple(o.shape)
        return self.head(o.float().mean(2).flatten(1)).squeeze(-1)

N_LAYERS = 22
FROZEN_PREFIX = ("backbone.to_patch_embedding.", "backbone.fourier4d.", "backbone.mlp4d.", "backbone.ln.", "backbone.cls_query_token") + \
                tuple(f"backbone.transformer.layers.{i}." for i in range(N_LAYERS // 2))
def new_rv(seed, freeze=False):
    torch.manual_seed(seed)
    m = RVNet().to(DEV)
    assert len(m.backbone.transformer.layers) == N_LAYERS
    if freeze:
        for n, p in m.named_parameters():
            if n.startswith(FROZEN_PREFIX):
                p.requires_grad_(False)
    return m

AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")

def predict_windows(model, Ws, bs=64):
    """Вероятность каждого окна: список массивов, по одному на запись."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for W in Ws:
            p = [torch.sigmoid(model(rv_input(W[b:b + bs]).to(DEV)).float()).cpu().numpy() for b in range(0, len(W), bs)]
            out.append(np.concatenate(p))
    return out

def predict(model, Ws):
    """Медиана вероятности по окнам для каждой записи (как в прошлом блокноте REVE)."""
    return np.array([np.median(p) for p in predict_windows(model, Ws)])

def rank(s): return rankdata(s) / len(s)
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

SOS_BP = ss.butter(4, [0.3, 75], btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(50, 30, fs=SFREQ)
def cb_filter(X):
    """0,3-75 Гц + режекция 50 Гц, как в блокноте CBraMod."""
    if X.shape[-1] < 60 * SFREQ:
        return X
    return ss.filtfilt(B_N, A_N, ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1), axis=-1)

def windows(X):
    n = X.shape[1] // L
    return X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)

def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор")

## 2. NMT и турецкий набор (подкачка обоих параллельно)

In [ ]:
ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).sort_values("recordname").reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
if SMOKE:
    lab = pd.concat([lab[(lab["loc"] == "train") & (lab.y == 0)].head(40), lab[(lab["loc"] == "train") & (lab.y == 1)].head(20),
                     lab[(lab["loc"] == "eval")].head(20)]).sort_values("recordname").reset_index(drop=True)
    tfiles = tfiles[:10] + tfiles[-10:]
print(pd.crosstab(lab["loc"], lab["label"]))

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time(); allf = list(lab.path) + [f for f, _ in tfiles]
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, allf)):
        if (i + 1) % 500 == 0 or i + 1 == len(allf):
            print(f"подкачано {i+1}/{len(allf)} {time.time()-t0:.0f} с", flush=True)

ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in EL]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

NMT_STORE_K = 100.0   # масштаб в файлах неверный; окна всё равно нормируются, множитель только для float16
def nmt_job(i, spread_all=False):
    """spread_all=True — до 60 окон равномерно и для записей train (нужно для отложенных 15 %)."""
    r = lab.iloc[i]
    try:
        raw = mne.io.read_raw_edf(r.path, preload=True)
        if raw.info["sfreq"] != SFREQ:
            raw.resample(SFREQ)
        names = {norm(c): c for c in raw.ch_names}
        X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * NMT_STORE_K
    except Exception:
        return r.recordname, None
    W = windows(cb_filter(X))
    if len(W) == 0:
        return r.recordname, None
    if r["loc"] == "train" and not spread_all and len(W) > CAP_TRAIN:
        W = W[np.sort(np.random.default_rng(i).choice(len(W), CAP_TRAIN, replace=False))]
    elif r["loc"] == "eval" or spread_all:
        W = W[spread(len(W), CAP_EVAL)]
    return r.recordname, W.astype(np.float16)

NMT = {}
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (name, res) in enumerate(ex.map(nmt_job, range(len(lab)))):
        if res is not None:
            NMT[name] = res
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT)].reset_index(drop=True)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))
print("окон: train", sum(len(NMT[n]) for n in tr.recordname), "| eval", sum(len(NMT[n]) for n in ev.recordname))

## 3. Турецкий набор: 19 каналов, средний референт, 200 Гц, окна 10 с подряд

In [ ]:
# индексы турецких каналов в порядке EL: Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 Fz Cz Pz
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]

def tr_job(f):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)                                                       # средний референт
    X = ss.resample_poly(X, 2, 5, axis=-1)                                                  # 200 Гц
    W = windows(cb_filter(np.concatenate(list(X), axis=-1)))
    return W[spread(len(W), CAP_EVAL)].astype(np.float16), float(np.std(X))

TR, TR_AMP = [], []
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for a, b in ex.map(tr_job, [f for f, _ in tfiles]):
        TR.append(a); TR_AMP.append(b)
y_tr = np.array([yy for _, yy in tfiles]); tr_ids = [os.path.basename(f) for f, _ in tfiles]
TR_AMP = np.array(TR_AMP)
print(f"Турция прочитано за {time.time()-t0:.0f} с | эпилепсия {int(y_tr.sum())}, здоровые {int((1-y_tr).sum())} | окон на запись: медиана {int(np.median([len(w) for w in TR]))}")
print("эталон перекоса: СКО сигнала записи как оценка, ROC AUC =", round(roc_auc_score(y_tr, TR_AMP), 3))

## 4. Раздел NMT train (как раньше) и отложенные 15 % по 60 окон

In [ ]:
fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
yw = np.concatenate([np.full(len(NMT[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])
w_cls = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum()); w_cls /= w_cls.sum()
Xw = np.concatenate([NMT[n] for n in fit_names])
val_W = [NMT[n] for n in val_names]          # 8 окон — для выбора эпохи (как в прошлом блокноте)
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор: {len(val_names)}")

# отложенные записи заново, до 60 окон равномерно — как NMT eval и Турция (для порога и выбора способа сборки)
pos = {n: i for i, n in enumerate(lab.recordname)}
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    VAL60 = dict(ex.map(lambda n: nmt_job(pos[n], spread_all=True), val_names))
assert all(VAL60[n] is not None for n in val_names)
VAL60 = [VAL60[n] for n in val_names]
print(f"отложенные по 60 окон: {sum(len(w) for w in VAL60)} окон, {time.time()-t0:.0f} с | осталось {left():.0f} мин")

## 5. Дообучение: выбор настройки по отложенным 15 %, затем 3 запуска

In [ ]:
NOISE, BS, MAX_EP, PATIENCE, LR_HEAD = 0.1, 32, (2 if SMOKE else 8), 2, 5e-4
def augment(x, g):
    B = x.shape[0]
    x = x + NOISE * torch.randn_like(x)
    m = torch.rand(B, device=x.device) < 0.5
    x[m] = x[m][:, MIRROR]
    for b in range(B):
        if g.random() < 0.5:
            x[b, g.choice(19, g.integers(1, 3), replace=False)] = 0
        x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)
    return x

def make_opt(model, total_steps, lr_back):
    back = [p for n, p in model.named_parameters() if n.startswith("backbone") and p.requires_grad]
    head = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
    opt = torch.optim.AdamW([{"params": back, "lr": lr_back}, {"params": head, "lr": LR_HEAD}], weight_decay=0.05)
    warm_n = max(1, int(0.05 * total_steps))
    f = lambda s: (s + 1) / warm_n if s < warm_n else 0.5 * (1 + np.cos(np.pi * (s - warm_n) / max(1, total_steps - warm_n))) * 0.99 + 0.01
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, f)

def step(model, opt, sched, scaler, x, t, params):
    with torch.autocast(**AMP):
        out = model(x)
    loss = F.binary_cross_entropy_with_logits(out.float(), t.float())
    bad = not torch.isfinite(loss)
    opt.zero_grad()
    if not bad:
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update()
    sched.step()
    return loss.item(), bad

CONFIGS = {"1e-5": dict(lr=1e-5, freeze=False), "3e-6": dict(lr=3e-6, freeze=False), "1e-5 заморожена половина": dict(lr=1e-5, freeze=True)}

def train_run(seed, cfg, tag, deadline=None):
    """До MAX_EP эпох; остановка, если отложенный ROC AUC не растёт PATIENCE эпох подряд (плато) или кончается время. Возвращается лучшая эпоха."""
    c = CONFIGS[cfg]; g = np.random.default_rng(seed)
    model = new_rv(seed, c["freeze"])
    params = [p for p in model.parameters() if p.requires_grad]
    steps_ep = int(np.ceil(len(yw) / BS))
    opt, sched = make_opt(model, MAX_EP * steps_ep, c["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best_auc, best_state, best_ep, hist, n_bad, stop = -1, None, 0, [], 0, "макс. эпох"
    t0 = time.time()
    for ep in range(MAX_EP):
        model.train(); losses = []
        idx = g.choice(len(yw), size=len(yw), replace=True, p=w_cls)
        for b in range(0, len(idx), BS):
            j = idx[b:b + BS]
            l, bad = step(model, opt, sched, scaler, augment(rv_input(Xw[j]).to(DEV), g), torch.tensor(yw[j], device=DEV), params)
            n_bad += bad; losses.append(l)
        auc = roc_auc_score(y_val, predict(model, val_W))
        hist.append((round(float(np.nanmean(losses)), 3), round(float(auc), 3)))
        print(f"  {tag} {cfg} запуск {seed} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {auc:.3f} | шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с | осталось {left():.0f} мин", flush=True)
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if ep + 1 - best_ep >= PATIENCE:
            stop = "плато"; break
        if deadline is not None and time.time() + (time.time() - t0) / (ep + 1) > deadline:
            stop = "время"; break
    print(f"  {tag} {cfg} запуск {seed}: остановка — {stop}, лучшая эпоха {best_ep}, {(time.time()-t0)/60:.1f} мин", flush=True)
    model.load_state_dict(best_state)
    return model, dict(cfg=cfg, best_ep=best_ep, best_auc=best_auc, hist=hist, n_bad=n_bad, stop=stop,
                       minutes=round((time.time() - t0) / 60, 1), min_per_ep=(time.time() - t0) / 60 / len(hist))

def final_predict(model):
    """Тестовые наборы — один раз, для каждого из 3 итоговых запусков."""
    t0 = time.time()
    P = {"nmt_val": predict_windows(model, VAL60), "nmt_eval": predict_windows(model, [NMT[n] for n in ev.recordname]), "turkey": predict_windows(model, TR)}
    return P, (time.time() - t0) / 60

# --- выбор настройки (запуск 0), остальное время резервируется под 2 запуска и итоговые предсказания ---
T_TRAIN, GRID, PRED_MIN = time.time(), {}, 3.0     # PRED_MIN — оценка итоговых предсказаний одного запуска, уточняется ниже
reserve = lambda run_min: 2 * run_min + 3 * PRED_MIN + 1
for cfg in CONFIGS:
    done = [v for v in GRID.values()]
    ep_min = np.mean([v["min_per_ep"] for v in done]) if done else 3.5
    best_run = max(done, key=lambda v: v["best_auc"])["minutes"] if done else 3 * ep_min
    need = (3 if cfg != "3e-6" else 4) * ep_min * (0.8 if CONFIGS[cfg]["freeze"] else 1.0)
    if done and not SMOKE and left() < need + reserve(max(best_run, 3 * ep_min)):
        print(f"ПРОПУСК {cfg}: не укладывается (осталось {left():.0f} мин, нужно ≈{need + reserve(max(best_run, 3 * ep_min)):.0f})"); continue
    deadline = END - 60 * reserve(max(best_run if done else 3 * ep_min, 3 * ep_min))
    model, info = train_run(0, cfg, "выбор", deadline=deadline)
    info["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    GRID[cfg] = info
    print(f"{cfg}: лучшая эпоха {info['best_ep']}, отложенные NMT {info['best_auc']:.3f}, {info['minutes']} мин | осталось {left():.0f} мин", flush=True)
    del model
    if DEV == "cuda": torch.cuda.empty_cache()
CFG = max(GRID, key=lambda k: GRID[k]["best_auc"])
print("выбрана настройка:", CFG)
pd.DataFrame([dict(config=k, lr_backbone=CONFIGS[k]["lr"], frozen_half=CONFIGS[k]["freeze"], best_epoch=v["best_ep"], val=round(v["best_auc"], 3),
                   stop=v["stop"], hist=v["hist"], nan_steps=v["n_bad"], minutes=v["minutes"]) for k, v in GRID.items()]) \
  .to_csv(f"{OUT}/reve_v2_lr_choice.csv", index=False)

# --- 3 запуска выбранной настройки (запуск 0 — из выбора); тестовые наборы — один раз ---
RUNS = []
for seed in [0, 1, 2]:
    if seed == 0:
        model = new_rv(0, CONFIGS[CFG]["freeze"]); model.load_state_dict(GRID[CFG]["state"]); info = GRID[CFG]
    else:
        n_left = 3 - seed   # сколько запусков ещё обучать, включая этот
        deadline = time.time() + 60 * (left() - n_left * PRED_MIN - 0.5) / n_left
        model, info = train_run(seed, CFG, "reve2", deadline=deadline)
    P, pm = final_predict(model); PRED_MIN = max(PRED_MIN, pm)
    s_val, s_ev, s_tr = (np.array([np.median(p) for p in P[k]]) for k in ["nmt_val", "nmt_eval", "turkey"])
    RUNS.append(dict(model="reve_v2", config=CFG, seed=seed, best_epoch=info["best_ep"], stop=info["stop"], val=round(info["best_auc"], 3), hist=info["hist"],
                     nan_steps=info["n_bad"], minutes=info["minutes"], val60=roc_auc_score(y_val, s_val),
                     nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr), P=P, s_val=s_val, s_ev=s_ev, s_tr=s_tr,
                     turkey_med_p0=float(np.median(s_tr[y_tr == 0])), turkey_med_p1=float(np.median(s_tr[y_tr == 1])), n_unique_tr=len(np.unique(s_tr))))
    print(f"reve2 запуск {seed}: эпоха {info['best_ep']} ({info['stop']}) | отложенные 60 окон {RUNS[-1]['val60']:.3f} | NMT eval {RUNS[-1]['nmt_eval']:.3f} | Турция {RUNS[-1]['turkey']:.3f} | "
          f"медиана p здоровые/эпилепсия {RUNS[-1]['turkey_med_p0']:.3f}/{RUNS[-1]['turkey_med_p1']:.3f} | разных значений {RUNS[-1]['n_unique_tr']}/{len(s_tr)} | "
          f"предсказания {pm:.1f} мин | осталось {left():.0f} мин", flush=True)
    del model
    if DEV == "cuda": torch.cuda.empty_cache()
del GRID
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин")
runs = pd.DataFrame([{k: v for k, v in r.items() if k not in ("P", "s_val", "s_ev", "s_tr")} for r in RUNS])
runs.to_csv(f"{OUT}/reve_v2_runs.csv", index=False)
print(runs.drop(columns="hist").round(3).to_string(index=False))

## 6. Итоги: ROC AUC, пороговые метрики (порог — по отложенным 15 % NMT train, на NMT eval и Турцию перенесён без изменений), csv по людям и по окнам

In [ ]:
y_ev = ev.y.to_numpy()
ens_ev = np.mean([rank(r["s_ev"]) for r in RUNS], 0); ens_tr = np.mean([rank(r["s_tr"]) for r in RUNS], 0); ens_val = np.mean([rank(r["s_val"]) for r in RUNS], 0)
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
summary = pd.DataFrame([
    {"модель": f"REVE v2 ({CFG}), запуски", "n": len(runs), "отложенные 15 %": f"{runs.val60.mean():.3f} ± {sd(runs.val60):.3f}",
     "NMT eval": f"{runs.nmt_eval.mean():.3f} ± {sd(runs.nmt_eval):.3f}",
     "Турция": f"{runs.turkey.mean():.3f} ± {sd(runs.turkey):.3f} ({' / '.join(f'{v:.2f}' for v in runs.turkey)})"},
    {"модель": f"REVE v2 ({CFG}), ансамбль запусков", "n": "", "отложенные 15 %": fmt(y_val, ens_val), "NMT eval": fmt(y_ev, ens_ev), "Турция": fmt(y_tr, ens_tr)}])
summary.to_csv(f"{OUT}/reve_v2_summary.csv", index=False)
print(summary.to_string(index=False))
print("Связь с амплитудой записи на Турции (Спирмен):", round(spearmanr(ens_tr, TR_AMP).correlation, 2))

# пороговые метрики: оценка — среднее вероятностей запусков (у рангов нет общего масштаба между наборами)
p_val = np.mean([r["s_val"] for r in RUNS], 0); p_ev = np.mean([r["s_ev"] for r in RUNS], 0); p_tr = np.mean([r["s_tr"] for r in RUNS], 0)
fpr, tpr, thr = roc_curve(y_val, p_val)
THR = {"Юден": float(thr[np.argmax(tpr - fpr)]), "чувствительность 80 %": float(thr[np.argmax(tpr >= 0.8)])}
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}-{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}
rows = []
for name, t in THR.items():
    for ds, y, s in [("отложенные 15 %", y_val, p_val), ("NMT eval", y_ev, p_ev), ("Турция", y_tr, p_tr)]:
        m = metr_ci(y, s, t)
        rows.append({"модель": "REVE v2", "порог": name, "значение порога": round(t, 3), "набор": ds,
                     "примечание": "порог: отложенные 15 % NMT train" + ("" if ds == "отложенные 15 %" else "; перенесён без изменений"), "n": len(y),
                     "ROC AUC": fmt(y, s), "accuracy": m["acc"], "сбаланс. accuracy": m["bacc"], "precision": m["prec"], "recall (чувств.)": m["rec"], "специфичность": m["spec"]})
thr_tab = pd.DataFrame(rows)
thr_tab.to_csv(f"{OUT}/reve_v2_threshold_metrics.csv", index=False)
print(thr_tab.drop(columns=["примечание"]).to_string(index=False))

# по людям: медиана по окнам для каждого запуска, ансамбль рангов и среднее вероятностей
parts = []
for ds, ids, y, amp in [("nmt_val", list(val_names), y_val, None), ("nmt_eval", ev.recordname.tolist(), y_ev, None), ("turkey", tr_ids, y_tr, TR_AMP)]:
    d = pd.DataFrame({"dataset": ds, "id": ids, "y": y})
    if amp is not None: d["amp"] = amp
    key = {"nmt_val": "s_val", "nmt_eval": "s_ev", "turkey": "s_tr"}[ds]
    for r in RUNS: d[f"reve2_seed{r['seed']}"] = r[key]
    d["reve2_ens_rank"] = np.mean([rank(r[key]) for r in RUNS], 0)
    d["reve2_ens_prob"] = np.mean([r[key] for r in RUNS], 0)
    parts.append(d)
pd.concat(parts).to_csv(f"{OUT}/reve_v2_per_person.csv", index=False)

# по окнам: набор, id человека, номер окна (по порядку взятых окон записи), запуск, вероятность
wparts = []
for ds, ids, y in [("nmt_val", list(val_names), y_val), ("nmt_eval", ev.recordname.tolist(), y_ev), ("turkey", tr_ids, y_tr)]:
    for r in RUNS:
        P = r["P"][ds]
        wparts.append(pd.DataFrame({"dataset": ds, "id": np.repeat(ids, [len(p) for p in P]), "y": np.repeat(y, [len(p) for p in P]),
                                    "window": np.concatenate([np.arange(len(p)) for p in P]), "seed": r["seed"], "p": np.concatenate(P).round(6)}))
wdf = pd.concat(wparts)
wdf.to_csv(f"{OUT}/reve_v2_window_scores.csv.gz", index=False, compression="gzip")
print(f"оценки окон: {len(wdf)} строк, {os.path.getsize(f'{OUT}/reve_v2_window_scores.csv.gz')/1e6:.1f} МБ")
print("файлы:", sorted(os.listdir(OUT)))
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

## 7. Вывод Kaggle (без строк о подкачке файлов и служебных предупреждений)

Тестовые наборы считались один раз. NaN/inf нет, у групп вероятности различны (119 разных значений из 121). Перед этим блокнот прошёл короткую проверку на процессоре Kaggle (`SMOKE = True`, отдельный блокнот `zakharsann/h1-reve-v2-smoke`: 60 записей NMT train, 20 eval, 20 турецких, 1 эпоха, 67 мин, GPU не тратился); её тестовые цифры для решений не использовались.

**Перебор сократился до контроля.** Чтение NMT в этот раз заняло **41 мин** (2477 с) вместо 15 мин в прошлых блокнотах — медленная подкачка набора на Kaggle; отложенные 15 % по 60 окон — ещё 7 мин. На обучение осталось 30 мин, поэтому по правилу «сокращать перебор, а не запуски» прошёл только контроль 1·10⁻⁵ (1 эпоха, остановлен по времени), **3·10⁻⁶ и вариант с замороженной половиной не запускались**. Выбранная настройка — 1·10⁻⁵ (единственная), то есть обучение то же, что в прошлом блокноте; новое — остановка на плато, честный порог и оценки окон.

```
позиции: (19, 3)
устройство: Tesla T4
label  abnormal  normal
loc                    
eval         90      95
train       325    1907
NMT прочитано 400/2417 394 с
NMT прочитано 800/2417 778 с
NMT прочитано 1200/2417 1159 с
NMT прочитано 1600/2417 1589 с
NMT прочитано 2000/2417 2007 с
NMT прочитано 2400/2417 2458 с
NMT прочитано 2417/2417 2477 с
label  abnormal  normal
loc                    
eval         90      95
train       325    1904
окон: train 17775 | eval 10406
Турция прочитано за 203 с | эпилепсия 50, здоровые 71 | окон на запись: медиана 60
эталон перекоса: СКО сигнала записи как оценка, ROC AUC = 0.764
окон для обучения: 15103 (патология 0.14) | записей на выбор: 335
отложенные по 60 окон: 18569 окон, 401 с | осталось 30 мин
  выбор 1e-5 запуск 0 | эпоха 1: потери 0.531 | отложенные NMT 0.879 | шагов с NaN/inf 0 | 199 с | осталось 27 мин
  выбор 1e-5 запуск 0: остановка — время, лучшая эпоха 1, 3.3 мин
1e-5: лучшая эпоха 1, отложенные NMT 0.879, 3.3 мин | осталось 27 мин
ПРОПУСК 3e-6: не укладывается (осталось 27 мин, нужно ≈43)
ПРОПУСК 1e-5 заморожена половина: не укладывается (осталось 27 мин, нужно ≈38)
выбрана настройка: 1e-5
reve2 запуск 0: эпоха 1 (время) | отложенные 60 окон 0.881 | NMT eval 0.878 | Турция 0.894 | медиана p здоровые/эпилепсия 0.163/0.415 | разных значений 119/121 | предсказания 2.2 мин | осталось 25 мин
  reve2 1e-5 запуск 1 | эпоха 1: потери 0.532 | отложенные NMT 0.879 | шагов с NaN/inf 0 | 197 с | осталось 22 мин
  reve2 1e-5 запуск 1 | эпоха 2: потери 0.397 | отложенные NMT 0.869 | шагов с NaN/inf 0 | 394 с | осталось 18 мин
  reve2 1e-5 запуск 1: остановка — время, лучшая эпоха 1, 6.6 мин
reve2 запуск 1: эпоха 1 (время) | отложенные 60 окон 0.887 | NMT eval 0.907 | Турция 0.870 | медиана p здоровые/эпилепсия 0.143/0.257 | разных значений 119/121 | предсказания 2.2 мин | осталось 16 мин
  reve2 1e-5 запуск 2 | эпоха 1: потери 0.534 | отложенные NMT 0.880 | шагов с NaN/inf 0 | 197 с | осталось 13 мин
  reve2 1e-5 запуск 2 | эпоха 2: потери 0.406 | отложенные NMT 0.871 | шагов с NaN/inf 0 | 394 с | осталось 10 мин
  reve2 1e-5 запуск 2 | эпоха 3: потери 0.341 | отложенные NMT 0.863 | шагов с NaN/inf 0 | 591 с | осталось 6 мин
  reve2 1e-5 запуск 2: остановка — плато, лучшая эпоха 1, 9.9 мин
reve2 запуск 2: эпоха 1 (плато) | отложенные 60 окон 0.885 | NMT eval 0.889 | Турция 0.878 | медиана p здоровые/эпилепсия 0.176/0.512 | разных значений 119/121 | предсказания 2.2 мин | осталось 4 мин
всё обучение: 26.3 мин
  model config  seed  best_epoch  stop   val  nan_steps  minutes  val60  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr
reve_v2   1e-5     0           1 время 0.879          0      3.3  0.881     0.878   0.894          0.163          0.415          119
reve_v2   1e-5     1           1 время 0.879          0      6.6  0.887     0.907   0.870          0.143          0.257          119
reve_v2   1e-5     2           1 плато 0.880          0      9.9  0.885     0.889   0.878          0.176          0.512          119
                           модель n   отложенные 15 %          NMT eval                             Турция
          REVE v2 (1e-5), запуски 3     0.884 ± 0.003     0.891 ± 0.015 0.880 ± 0.012 (0.89 / 0.87 / 0.88)
REVE v2 (1e-5), ансамбль запусков   0.888 (0.84-0.93) 0.898 (0.85-0.94)                  0.889 (0.82-0.95)
Связь с амплитудой записи на Турции (Спирмен): 0.36
 модель                 порог  значение порога           набор   n           ROC AUC         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
REVE v2                  Юден            0.665 отложенные 15 % 335 0.889 (0.84-0.93) 0.83 (0.79-0.87)  0.82 (0.75-0.88) 0.46 (0.35-0.57) 0.80 (0.68-0.91) 0.84 (0.80-0.88)
REVE v2                  Юден            0.665        NMT eval 185 0.896 (0.85-0.94) 0.79 (0.73-0.84)  0.79 (0.73-0.84) 0.85 (0.76-0.93) 0.69 (0.59-0.79) 0.88 (0.82-0.95)
REVE v2                  Юден            0.665          Турция 121 0.890 (0.83-0.95) 0.62 (0.54-0.70)  0.54 (0.51-0.58) 1.00 (1.00-1.00) 0.08 (0.02-0.16) 1.00 (1.00-1.00)
REVE v2 чувствительность 80 %            0.615 отложенные 15 % 335 0.889 (0.84-0.93) 0.79 (0.74-0.83)  0.80 (0.74-0.86) 0.39 (0.29-0.49) 0.82 (0.70-0.92) 0.78 (0.73-0.83)
REVE v2 чувствительность 80 %            0.615        NMT eval 185 0.896 (0.85-0.94) 0.79 (0.74-0.85)  0.79 (0.73-0.85) 0.81 (0.72-0.89) 0.76 (0.66-0.84) 0.83 (0.75-0.91)
REVE v2 чувствительность 80 %            0.615          Турция 121 0.890 (0.83-0.95) 0.67 (0.59-0.75)  0.60 (0.55-0.66) 1.00 (1.00-1.00) 0.20 (0.10-0.32) 1.00 (1.00-1.00)
оценки окон: 108645 строк, 0.7 МБ
файлы: ['__notebook__.ipynb', 'reve_v2_lr_choice.csv', 'reve_v2_per_person.csv', 'reve_v2_runs.csv', 'reve_v2_summary.csv', 'reve_v2_threshold_metrics.csv', 'reve_v2_window_scores.csv.gz']
всё время блокнота: 80.4 мин
```

## 8. Сборка оценки записи из оценок окон (локально, без GPU, по `results/reve_v2_window_scores.csv.gz`)

Для каждого запуска оценка записи собирается из вероятностей её окон пятью способами; затем ансамбль 3 запусков — среднее рангов (как везде). **Способ выбирается по отложенным 15 % NMT train** (по 60 окон на запись, как в тесте); NMT eval и Турция только показываются. Верхние 10 % / 20 % — среднее ⌈0,1·n⌉ / ⌈0,2·n⌉ самых высоких окон записи (при 60 окнах — 6 / 12).

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr

RES = "../results"
w = pd.read_csv(f"{RES}/reve_v2_window_scores.csv.gz")
pp2 = pd.read_csv(f"{RES}/reve_v2_per_person.csv")
seeds = sorted(w.seed.unique())
top = lambda q: (lambda v: np.sort(v.to_numpy())[::-1][:int(np.ceil(q * len(v)))].mean())
AGG = {"медиана": "median", "среднее": "mean", "среднее верхних 10 %": top(0.1), "среднее верхних 20 %": top(0.2), "максимум": "max"}
rk = lambda s: rankdata(s) / len(s)

def boot(y, f, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: v.append(f(i))
    return np.percentile(v, [2.5, 97.5])

PER = {}   # способ -> таблица по людям: dataset, id, y, вероятность каждого запуска, ранг ансамбля, среднее вероятностей
for name, f in AGG.items():
    a = w.groupby(["dataset", "id", "seed"]).p.agg(f).unstack("seed")
    a.columns = [f"seed{s}" for s in a.columns]
    a = a.join(w.groupby(["dataset", "id"]).y.first()).reset_index()
    a["ens_rank"] = a.groupby("dataset")[[f"seed{s}" for s in seeds]].transform(lambda c: rk(c)).mean(1)
    a["ens_prob"] = a[[f"seed{s}" for s in seeds]].mean(1)
    PER[name] = a

# проверка: медиана по окнам совпадает с оценками блокнота Kaggle
chk = PER["медиана"].merge(pp2, on=["dataset", "id"], suffixes=("", "_k"))
assert len(chk) == len(pp2) and all(np.allclose(chk[f"seed{s}"], chk[f"reve2_seed{s}"], atol=1e-5) for s in seeds)

rows = []
for name, a in PER.items():
    row = {"способ сборки": name}
    for ds, lab in [("nmt_val", "отложенные 15 % (выбор)"), ("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        d = a[a.dataset == ds]; y, s = d.y.to_numpy(), d.ens_rank.to_numpy()
        lo, hi = boot(y, lambda i: roc_auc_score(y[i], s[i]))
        row[lab] = f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"; row[f"_{ds}"] = roc_auc_score(y, s)
    rows.append(row)
agg_tab = pd.DataFrame(rows)
AGG_BEST = agg_tab.loc[agg_tab._nmt_val.idxmax(), "способ сборки"]
print(agg_tab.drop(columns=[c for c in agg_tab if c.startswith("_")]).to_string(index=False))
print("\nвыбран по отложенным 15 %:", AGG_BEST)

print("\nПарный бутстреп разности ROC AUC с медианой (95% ДИ):")
for name in AGG:
    if name == "медиана": continue
    for ds, lab in [("nmt_val", "отложенные"), ("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        a, b = PER[name][PER[name].dataset == ds], PER["медиана"][PER["медиана"].dataset == ds]
        assert (a.id.to_numpy() == b.id.to_numpy()).all()
        y = a.y.to_numpy(); sa, sb = a.ens_rank.to_numpy(), b.ens_rank.to_numpy()
        f = lambda i: roc_auc_score(y[i], sa[i]) - roc_auc_score(y[i], sb[i]); lo, hi = boot(y, f)
        agg_tab.loc[agg_tab["способ сборки"] == name, f"Δ к медиане, {lab}"] = f"{f(np.arange(len(y))):+.3f} ({lo:+.3f}…{hi:+.3f})"
    print(f"  {name}: " + " | ".join(f"{c.split(', ')[1]} {agg_tab.loc[agg_tab['способ сборки'] == name, c].iloc[0]}" for c in agg_tab if c.startswith("Δ")))
agg_tab.drop(columns=[c for c in agg_tab if c.startswith("_")]).to_csv(f"{RES}/reve_v2_aggregation.csv", index=False)

# пороговые метрики для выбранного способа (если это не медиана): порог по отложенным 15 %, перенесён без изменений
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}-{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}
if AGG_BEST != "медиана":
    a = PER[AGG_BEST]; V, E, T = (a[a.dataset == d] for d in ["nmt_val", "nmt_eval", "turkey"])
    fpr, tpr, thr = roc_curve(V.y, V.ens_prob)
    THR = {"Юден": float(thr[np.argmax(tpr - fpr)]), "чувствительность 80 %": float(thr[np.argmax(tpr >= 0.8)])}
    rows = []
    for tn_, t in THR.items():
        for ds, D in [("отложенные 15 %", V), ("NMT eval", E), ("Турция", T)]:
            m = metr_ci(D.y.to_numpy(), D.ens_prob.to_numpy(), t)
            rows.append({"модель": f"REVE v2, {AGG_BEST}", "порог": tn_, "значение порога": round(t, 3), "набор": ds, "n": len(D),
                         "accuracy": m["acc"], "сбаланс. accuracy": m["bacc"], "precision": m["prec"], "recall (чувств.)": m["rec"], "специфичность": m["spec"]})
    thr_best = pd.DataFrame(rows); thr_best.to_csv(f"{RES}/reve_v2_threshold_metrics_{'top' if 'верх' in AGG_BEST else 'agg'}.csv", index=False)
    print(thr_best.to_string(index=False))

Вывод (выполнено локально по csv, без GPU):

```
способ сборки отложенные 15 % (выбор)          NMT eval            Турция
             медиана       0.888 (0.84-0.93) 0.898 (0.85-0.94) 0.889 (0.82-0.95)
             среднее       0.893 (0.85-0.94) 0.898 (0.85-0.94) 0.908 (0.85-0.96)
среднее верхних 10 %       0.910 (0.87-0.95) 0.926 (0.89-0.96) 0.922 (0.86-0.96)
среднее верхних 20 %       0.905 (0.86-0.94) 0.925 (0.89-0.96) 0.930 (0.88-0.97)
            максимум       0.915 (0.87-0.95) 0.932 (0.89-0.96) 0.847 (0.77-0.91)

выбран по отложенным 15 %: максимум

Парный бутстреп разности ROC AUC с медианой (95% ДИ):
  среднее: отложенные +0.005 (-0.000…+0.011) | NMT eval +0.000 (-0.006…+0.005) | Турция +0.018 (+0.004…+0.037)
  среднее верхних 10 %: отложенные +0.022 (-0.002…+0.047) | NMT eval +0.029 (+0.006…+0.053) | Турция +0.032 (-0.017…+0.087)
  среднее верхних 20 %: отложенные +0.017 (-0.001…+0.035) | NMT eval +0.027 (+0.008…+0.048) | Турция +0.041 (+0.003…+0.084)
  максимум: отложенные +0.026 (-0.002…+0.056) | NMT eval +0.034 (+0.005…+0.063) | Турция -0.042 (-0.116…+0.032)
           модель                 порог  значение порога           набор   n         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
REVE v2, максимум                  Юден            0.903 отложенные 15 % 335 0.87 (0.84-0.91)  0.84 (0.78-0.90) 0.55 (0.43-0.66) 0.80 (0.68-0.90) 0.89 (0.85-0.92)
REVE v2, максимум                  Юден            0.903        NMT eval 185 0.84 (0.78-0.89)  0.84 (0.78-0.89) 0.89 (0.82-0.96) 0.76 (0.66-0.84) 0.92 (0.85-0.97)
REVE v2, максимум                  Юден            0.903          Турция 121 0.60 (0.52-0.69)  0.52 (0.50-0.55) 1.00 (1.00-1.00) 0.04 (0.00-0.10) 1.00 (1.00-1.00)
REVE v2, максимум чувствительность 80 %            0.889 отложенные 15 % 335 0.85 (0.81-0.89)  0.84 (0.78-0.89) 0.49 (0.39-0.61) 0.82 (0.71-0.93) 0.86 (0.82-0.90)
REVE v2, максимум чувствительность 80 %            0.889        NMT eval 185 0.85 (0.80-0.90)  0.85 (0.80-0.90) 0.90 (0.82-0.96) 0.79 (0.70-0.87) 0.92 (0.85-0.97)
REVE v2, максимум чувствительность 80 %            0.889          Турция 121 0.60 (0.52-0.69)  0.52 (0.50-0.55) 1.00 (1.00-1.00) 0.04 (0.00-0.10) 1.00 (1.00-1.00)
```

## 9. Сравнение со старой REVE, EEG-DINO и CBraMod; ансамбль с EEG-DINO; калибровка на месте (локально, по csv по людям)

Основа — `results/eegdino_ensembles_per_person.csv` (общие записи: NMT eval 184 + Турция 121; там же старая REVE, EEG-DINO, CBraMod). REVE v2 — медиана по окнам, ансамбль рангов 3 запусков (как старая REVE). Калибровка — та же функция `calib`, seed 0 и тот же порядок людей, что в `results/eegdino_turkey_local_threshold_calibration.csv`; совпадение общих строк с той таблицей проверяется.

In [ ]:
dn = pd.read_csv(f"{RES}/eegdino_ensembles_per_person.csv")
m = dn.merge(pp2[["dataset", "id", "y", "reve2_ens_rank"]], on=["dataset", "id"], suffixes=("", "_v2"), how="left")
assert m.reve2_ens_rank.notna().all() and (m.y == m.y_v2).all()
if AGG_BEST != "медиана":
    m = m.merge(PER[AGG_BEST][["dataset", "id", "ens_rank"]].rename(columns={"ens_rank": "reve2_best"}), on=["dataset", "id"], how="left")
print({d: int((m.dataset == d).sum()) for d in ["nmt_eval", "turkey"]}, "общих записей")

SC = {}
for ds in ["nmt_eval", "turkey"]:
    d = m[m.dataset == ds]
    s = {"REVE v2": rk(d.reve2_ens_rank), "REVE (старая)": rk(d.reve_rank), "EEG-DINO": d.eegdino_rank.to_numpy(), "CBraMod": rk(d.cbramod_rank)}   # EEG-DINO — средний ранг 3 запусков без повторного ранжирования, как в его блокноте
    if AGG_BEST != "медиана": s[f"REVE v2 ({AGG_BEST})"] = rk(d.reve2_best)
    s["REVE v2 + EEG-DINO"] = (s["REVE v2"] + s["EEG-DINO"]) / 2
    s["REVE (старая) + EEG-DINO"] = d.eegdino_reve_rank.to_numpy()   # как в eegdino_ensembles_per_person.csv
    SC[ds] = (d.y.to_numpy(), s, d)

rows = []
for name in SC["turkey"][1]:
    row = {"модель / ансамбль (средние ранги)": name}
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]; lo, hi = boot(y, lambda i: roc_auc_score(y[i], s[name][i]))
        row[lab] = f"{roc_auc_score(y, s[name]):.3f} ({lo:.2f}-{hi:.2f})"
    rows.append(row)
summ = pd.DataFrame(rows)
print(summ.to_string(index=False))

print("\nПарный бутстреп разности ROC AUC (95% ДИ):")
pairs = [("REVE v2", "REVE (старая)"), ("REVE v2", "EEG-DINO"), ("REVE v2", "CBraMod"),
         ("REVE v2 + EEG-DINO", "REVE v2"), ("REVE v2 + EEG-DINO", "EEG-DINO"), ("REVE v2 + EEG-DINO", "REVE (старая) + EEG-DINO")]
if AGG_BEST != "медиана": pairs.insert(1, (f"REVE v2 ({AGG_BEST})", "REVE (старая)"))
diffs = []
for a, b in pairs:
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]
        f = lambda i: roc_auc_score(y[i], s[a][i]) - roc_auc_score(y[i], s[b][i])
        lo, hi = boot(y, f); dv = f(np.arange(len(y)))
        diffs.append({"разность": f"{a} − {b}", "набор": lab, "ΔROC AUC": round(dv, 3), "ДИ": f"{lo:+.3f}…{hi:+.3f}"})
        print(f"  {a} − {b} | {lab}: {dv:+.3f} ({lo:+.3f}…{hi:+.3f})")
y, s, d = SC["turkey"]
print("\nСпирмен на Турции, REVE v2 с:", " | ".join(f"{k} {spearmanr(s['REVE v2'], v).correlation:.2f}" for k, v in
      {"амплитуда": pp2[pp2.dataset == "turkey"].set_index("id").loc[d.id, "amp"].to_numpy(), "REVE (старая)": s["REVE (старая)"],
       "EEG-DINO": s["EEG-DINO"], "CBraMod": s["CBraMod"]}.items()))

# --- калибровка на месте: тот же код, что в блокноте EEG-DINO ---
def calib(y, sc, k, reps=500, seed=0):
    r = np.random.default_rng(seed); i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]; M = {n: [] for n in sc}
    for _ in range(reps):
        c = np.concatenate([r.choice(i1, k // 2, replace=False), r.choice(i0, k // 2, replace=False)])
        rest = np.setdiff1d(np.arange(len(y)), c); yy = y[rest]
        for n, s in sc.items():
            fpr, tpr, th = roc_curve(y[c], s[c]); pr = s[rest] >= th[np.argmax(tpr - fpr)]
            tp = (pr & (yy == 1)).sum(); tn = (~pr & (yy == 0)).sum(); fp = (pr & (yy == 0)).sum(); fn = (~pr & (yy == 1)).sum()
            rec, spec = tp / (tp + fn), tn / (tn + fp)
            M[n].append(dict(rec=rec, spec=spec, bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, acc=(tp + tn) / len(yy)))
    out = []
    for n in sc:
        D = pd.DataFrame(M[n]); f = lambda c: f"{D[c].mean():.2f} ({np.nanpercentile(D[c], 2.5):.2f}–{np.nanpercentile(D[c], 97.5):.2f})"
        out.append({"калибровка": f"{k} чел.", "модель": n, "recall": f("rec"), "специфичность": f("spec"), "сбаланс. accuracy": f("bacc"),
                    "precision": f("prec"), "accuracy": f("acc")})
    return out
cal_sc = {"REVE v2": s["REVE v2"], "REVE": s["REVE (старая)"], "EEG-DINO": s["EEG-DINO"], "CBraMod": s["CBraMod"],
          "REVE v2 + EEG-DINO": s["REVE v2 + EEG-DINO"], "EEG-DINO + REVE": s["REVE (старая) + EEG-DINO"]}
cal = pd.DataFrame([row for k in [10, 20, 30] for row in calib(y, cal_sc, k)])
print("\nКалибровка на месте (среднее по 500 повторам, 2,5–97,5 % повторов):")
print(cal.to_string(index=False))

old = pd.read_csv(f"{RES}/eegdino_turkey_local_threshold_calibration.csv")
chk = cal.merge(old, on=["калибровка", "модель"], suffixes=("", "_old"))
cols = ["recall", "специфичность", "сбаланс. accuracy", "precision", "accuracy"]
same = all((chk[c] == chk[c + "_old"]).all() for c in cols)
print(f"\nсовпадение с eegdino_turkey_local_threshold_calibration.csv по {len(chk)} общим строкам ({sorted(set(chk['модель']))}):", same)
assert same and len(chk) == 12
cal["модель"] = cal["модель"].replace({"REVE": "REVE (старая)", "EEG-DINO + REVE": "REVE (старая) + EEG-DINO"})

summ.to_csv(f"{RES}/reve_v2_ensembles_summary.csv", index=False)
pd.DataFrame(diffs).to_csv(f"{RES}/reve_v2_paired_differences.csv", index=False)
cal.to_csv(f"{RES}/reve_v2_turkey_local_threshold_calibration.csv", index=False)
out = []
for ds in ["nmt_eval", "turkey"]:
    yy, s, d = SC[ds]; t = d[["dataset", "id", "y"]].copy()
    for n, v in s.items(): t[n.replace(" + ", "_").replace(" ", "").replace("(", "").replace(")", "").replace("-", "").replace("%", "").lower() + "_rank"] = v
    out.append(t)
pd.concat(out).to_csv(f"{RES}/reve_v2_ensembles_per_person.csv", index=False)

Вывод (выполнено локально по csv, без GPU):

```
{'nmt_eval': 184, 'turkey': 121} общих записей
модель / ансамбль (средние ранги)          NMT eval            Турция
                          REVE v2 0.902 (0.86-0.94) 0.889 (0.82-0.95)
                    REVE (старая) 0.879 (0.83-0.92) 0.874 (0.80-0.93)
                         EEG-DINO 0.878 (0.83-0.92) 0.856 (0.79-0.91)
                          CBraMod 0.845 (0.79-0.90) 0.820 (0.74-0.88)
               REVE v2 (максимум) 0.931 (0.90-0.96) 0.847 (0.77-0.91)
               REVE v2 + EEG-DINO 0.904 (0.86-0.94) 0.907 (0.84-0.96)
         REVE (старая) + EEG-DINO 0.893 (0.84-0.93) 0.896 (0.83-0.95)

Парный бутстреп разности ROC AUC (95% ДИ):
  REVE v2 − REVE (старая) | NMT eval: +0.023 (+0.010…+0.037)
  REVE v2 − REVE (старая) | Турция: +0.015 (-0.003…+0.037)
  REVE v2 (максимум) − REVE (старая) | NMT eval: +0.052 (+0.021…+0.088)
  REVE v2 (максимум) − REVE (старая) | Турция: -0.027 (-0.102…+0.052)
  REVE v2 − EEG-DINO | NMT eval: +0.024 (-0.010…+0.063)
  REVE v2 − EEG-DINO | Турция: +0.033 (-0.025…+0.096)
  REVE v2 − CBraMod | NMT eval: +0.057 (+0.018…+0.098)
  REVE v2 − CBraMod | Турция: +0.070 (+0.003…+0.142)
  REVE v2 + EEG-DINO − REVE v2 | NMT eval: +0.002 (-0.014…+0.019)
  REVE v2 + EEG-DINO − REVE v2 | Турция: +0.018 (-0.010…+0.049)
  REVE v2 + EEG-DINO − EEG-DINO | NMT eval: +0.026 (+0.006…+0.051)
  REVE v2 + EEG-DINO − EEG-DINO | Турция: +0.051 (+0.014…+0.093)
  REVE v2 + EEG-DINO − REVE (старая) + EEG-DINO | NMT eval: +0.012 (+0.005…+0.020)
  REVE v2 + EEG-DINO − REVE (старая) + EEG-DINO | Турция: +0.011 (-0.002…+0.027)

Спирмен на Турции, REVE v2 с: амплитуда 0.36 | REVE (старая) 0.98 | EEG-DINO 0.69 | CBraMod 0.69

Калибровка на месте (среднее по 500 повторам, 2,5–97,5 % повторов):
калибровка             модель           recall    специфичность сбаланс. accuracy        precision         accuracy
   10 чел.            REVE v2 0.65 (0.27–0.93) 0.89 (0.55–1.00)  0.77 (0.63–0.84) 0.87 (0.57–1.00) 0.79 (0.67–0.86)
   10 чел.               REVE 0.62 (0.24–0.91) 0.88 (0.50–1.00)  0.75 (0.62–0.81) 0.85 (0.55–1.00) 0.78 (0.66–0.85)
   10 чел.           EEG-DINO 0.64 (0.22–0.96) 0.80 (0.56–1.00)  0.72 (0.61–0.78) 0.73 (0.59–1.00) 0.74 (0.68–0.77)
   10 чел.            CBraMod 0.61 (0.18–0.96) 0.76 (0.36–1.00)  0.69 (0.59–0.74) 0.70 (0.51–1.00) 0.70 (0.60–0.74)
   10 чел. REVE v2 + EEG-DINO 0.67 (0.26–0.93) 0.91 (0.58–1.00)  0.79 (0.63–0.85) 0.87 (0.60–1.00) 0.81 (0.67–0.86)
   10 чел.    EEG-DINO + REVE 0.65 (0.27–0.93) 0.90 (0.55–1.00)  0.77 (0.63–0.83) 0.86 (0.58–1.00) 0.80 (0.67–0.86)
   20 чел.            REVE v2 0.67 (0.40–0.90) 0.90 (0.54–1.00)  0.79 (0.69–0.85) 0.88 (0.56–1.00) 0.81 (0.68–0.87)
   20 чел.               REVE 0.65 (0.38–0.90) 0.88 (0.49–1.00)  0.77 (0.66–0.82) 0.84 (0.54–1.00) 0.79 (0.65–0.86)
   20 чел.           EEG-DINO 0.72 (0.30–0.96) 0.75 (0.54–0.98)  0.74 (0.63–0.78) 0.68 (0.57–0.92) 0.74 (0.69–0.78)
   20 чел.            CBraMod 0.66 (0.25–0.95) 0.73 (0.39–1.00)  0.69 (0.61–0.74) 0.66 (0.51–1.00) 0.70 (0.61–0.74)
   20 чел. REVE v2 + EEG-DINO 0.73 (0.45–0.93) 0.89 (0.75–1.00)  0.81 (0.72–0.85) 0.84 (0.69–1.00) 0.83 (0.77–0.87)
   20 чел.    EEG-DINO + REVE 0.70 (0.42–0.90) 0.89 (0.57–0.98)  0.79 (0.69–0.84) 0.84 (0.59–0.97) 0.81 (0.71–0.87)
   30 чел.            REVE v2 0.69 (0.49–0.90) 0.90 (0.54–1.00)  0.79 (0.71–0.86) 0.86 (0.55–1.00) 0.82 (0.68–0.89)
   30 чел.               REVE 0.67 (0.43–0.91) 0.87 (0.46–1.00)  0.77 (0.68–0.83) 0.82 (0.51–1.00) 0.79 (0.63–0.87)
   30 чел.           EEG-DINO 0.75 (0.40–0.97) 0.73 (0.54–0.95)  0.74 (0.66–0.79) 0.65 (0.56–0.82) 0.74 (0.68–0.79)
   30 чел.            CBraMod 0.68 (0.26–0.94) 0.72 (0.51–1.00)  0.70 (0.62–0.76) 0.64 (0.53–1.00) 0.70 (0.65–0.76)
   30 чел. REVE v2 + EEG-DINO 0.76 (0.49–0.93) 0.87 (0.73–1.00)  0.81 (0.72–0.86) 0.81 (0.67–1.00) 0.83 (0.78–0.87)
   30 чел.    EEG-DINO + REVE 0.72 (0.49–0.91) 0.87 (0.65–0.98)  0.80 (0.72–0.84) 0.81 (0.61–0.96) 0.81 (0.73–0.87)

совпадение с eegdino_turkey_local_threshold_calibration.csv по 12 общим строкам (['CBraMod', 'EEG-DINO', 'EEG-DINO + REVE', 'REVE']): True
```

## 10. Итоги

### Настройка и эпохи
- Выбор скорости основы **не состоялся**: из-за 41 мин чтения NMT проверен только контроль 1·10⁻⁵ (отложенные 15 %, 8 окон: 0,879 после 1-й эпохи). 3·10⁻⁶ и 1·10⁻⁵ с замороженной первой половиной — не запускались. Вопрос «помогает ли мягче обучение» остаётся открытым.
- 3 запуска с 1·10⁻⁵: **лучшая эпоха — снова первая во всех трёх** (seed 0 — остановлен по времени после 1 эпохи; seed 1 — по времени после 2: 0,879 → 0,869; seed 2 — плато после 3: 0,880 → 0,871 → 0,863). Быстрое переобучение подтверждается.

### ROC AUC (95% ДИ), медиана по окнам, ансамбль рангов 3 запусков
| Модель | Отложенные 15 % (335) | NMT eval | Турция |
|---|---|---|---|
| **REVE v2** (все 185 записей NMT eval) | 0,888 (0,84–0,93) | 0,898 (0,85–0,94) | **0,889 (0,82–0,95)** |
| REVE v2 (общие 184 записи) | | 0,902 (0,86–0,94) | 0,889 (0,82–0,95) |
| REVE старая | — | 0,879 (0,83–0,92) | 0,874 (0,80–0,93) |
| EEG-DINO | — | 0,878 (0,83–0,92) | 0,856 (0,79–0,91) |
| CBraMod | — | 0,845 (0,79–0,90) | 0,820 (0,74–0,88) |
| REVE v2 + EEG-DINO | — | 0,904 (0,86–0,94) | **0,907 (0,84–0,96)** |
| REVE старая + EEG-DINO | — | 0,893 (0,84–0,93) | 0,896 (0,83–0,95) |

Запуски: NMT eval 0,88 / 0,91 / 0,89, Турция 0,89 / 0,87 / 0,88.

Парные разности (95% ДИ): REVE v2 − старая: NMT eval **+0,023 (+0,010…+0,037)**, Турция +0,015 (−0,003…+0,037). Это **не эффект новой настройки** — скорость та же, оценки со старой REVE почти совпадают (Спирмен 0,98 на Турции); отличие — другие запуски и немного другой ход скорости (косинус рассчитан на 8 эпох, а не 6). Скорее разброс между запусками. REVE v2 − EEG-DINO: +0,024 / +0,033, не значимо. REVE v2 − CBraMod: NMT eval +0,057 (+0,018…+0,098), Турция +0,070 (+0,003…+0,142). REVE v2 + EEG-DINO − REVE v2: +0,002 / +0,018 (−0,010…+0,049), не значимо; − EEG-DINO: Турция +0,051 (+0,014…+0,093).

### Честные пороговые метрики (среднее вероятностей 3 запусков, медиана по окнам; порог по отложенным 15 %, перенесён без изменений)
| Порог | Набор | Accuracy | Сбаланс. acc. | Precision | Recall | Специфичность |
|---|---|---|---|---|---|---|
| Юден 0,665 | отложенные (335) | 0,83 (0,79–0,87) | 0,82 (0,75–0,88) | 0,46 (0,35–0,57) | 0,80 (0,68–0,91) | 0,84 (0,80–0,88) |
| | NMT eval (185) | 0,79 (0,73–0,84) | 0,79 (0,73–0,84) | 0,85 (0,76–0,93) | 0,69 (0,59–0,79) | 0,88 (0,82–0,95) |
| | Турция (121) | 0,62 (0,54–0,70) | 0,54 (0,51–0,58) | 1,00 | **0,08 (0,02–0,16)** | 1,00 |
| чувств. 80 % 0,615 | отложенные | 0,79 (0,74–0,83) | 0,80 (0,74–0,86) | 0,39 (0,29–0,49) | 0,82 (0,70–0,92) | 0,78 (0,73–0,83) |
| | NMT eval | 0,79 (0,74–0,85) | 0,79 (0,73–0,85) | 0,81 (0,72–0,89) | 0,76 (0,66–0,84) | 0,83 (0,75–0,91) |
| | Турция | 0,67 (0,59–0,75) | 0,60 (0,55–0,66) | 1,00 | **0,20 (0,10–0,32)** | 1,00 |

На NMT eval перенесённый порог работает (сбаланс. accuracy 0,79). На Турцию порог NMT **не переносится**: вероятности там ниже (медиана у эпилепсии 0,26–0,51 при пороге 0,62–0,67). Precision на отложенных низкая (0,39–0,46) из-за доли патологии 15 % в NMT train.

### Сборка оценки записи из окон
По отложенным 15 % лучший — **максимум** (0,915 против 0,888 у медианы; разность +0,026, ДИ −0,002…+0,056). На NMT eval максимум тоже лучше (0,932, +0,034, ДИ +0,005…+0,063), но **на Турции хуже** (0,847, −0,042, ДИ −0,116…+0,032). Устойчивее всего **среднее верхних 10–20 % окон**: отложенные 0,910 / 0,905, NMT eval 0,926 / 0,925, Турция 0,922 / 0,930 (для 20 %: +0,041 к медиане, ДИ +0,003…+0,084). Правило «выбирать по отложенным» дало максимум — его и фиксируем как честный результат. Верхние 10–20 % — кандидат на следующий раз: выбирать между ними нужно заранее или на других данных, не по Турции. Пороговые метрики для максимума — `reve_v2_threshold_metrics_agg.csv`: на NMT eval сбаланс. accuracy 0,84–0,85, на Турции перенос снова не работает (recall 0,04).

### Калибровка на месте (Турция, Юден по k людям, 500 повторов; те же выборки, что в `eegdino_turkey_local_threshold_calibration.csv` — общие 12 строк совпали)
| k | REVE v2 | REVE старая | EEG-DINO | REVE v2 + EEG-DINO | REVE старая + EEG-DINO |
|---|---|---|---|---|---|
| 10 | 0,77 (0,63–0,84) | 0,75 | 0,72 | **0,79 (0,63–0,85)** | 0,77 |
| 20 | 0,79 (0,69–0,85) | 0,77 | 0,74 | **0,81 (0,72–0,85)** | 0,79 |
| 30 | 0,79 (0,71–0,86) | 0,77 | 0,74 | **0,81 (0,72–0,86)** | 0,80 |

(сбалансированная accuracy, среднее и 2,5–97,5 % повторов). При 20 людях REVE v2 + EEG-DINO: recall 0,73, специфичность 0,89, precision 0,84. Прирост к прежнему лучшему ансамблю +0,01–0,02 — в пределах разброса.

### Оговорки
- Связь с амплитудой записи на Турции: Спирмен +0,36 (старая REVE +0,31; одна амплитуда даёт 0,76).
- Турция — 121 человек, ДИ широкие; NMT («патология») и Турция («эпилепсия против здоровых») — разные задачи.
- REVE предобучена в том числе на TUH — для будущей проверки на TUEP нужно исключать пересечение.

**Затраты:** основной блокнот 80,4 мин на T4 (чтение данных ≈51 мин, обучение с предсказаниями 26 мин) ≈ **1,34 ч GPU**, в бюджет 1,5 ч уложились. Недельная квота за это время обнулилась (новая неделя с 24.10), поэтому по ней расход не сверить; оценка — по времени блокнота. Проверка на процессоре — 67 мин, без GPU.

Файлы: `results/reve_v2_{runs,lr_choice,summary,per_person,threshold_metrics}.csv` (Kaggle), `results/reve_v2_window_scores.csv.gz` (108 645 оценок окон: отложенные 15 %, NMT eval, Турция × 3 запуска), `results/reve_v2_aggregation.csv`, `results/reve_v2_threshold_metrics_agg.csv` (максимум), `results/reve_v2_ensembles_{summary,per_person}.csv`, `results/reve_v2_paired_differences.csv`, `results/reve_v2_turkey_local_threshold_calibration.csv`. Веса не сохранялись.